# Module 09 · Unit 01
# Finite Automata

| | |
|---|---|
| **Estimated time** | 55–65 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Protocol Verification \[PV\] |
| **Refresher** | Module 04 · Unit 02 — Directed Graphs (automata are directed graphs with labelled edges) |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Define a **Deterministic Finite Automaton (DFA)** as a 5-tuple and trace any string through it
2. Define a **Non-Deterministic Finite Automaton (NFA)** and explain what non-determinism means
3. Apply the **Subset Construction** to convert an NFA to an equivalent DFA
4. Identify the **regular language** recognised by a finite automaton
5. Explain why finite automata are the correct model for systems with **bounded memory**
6. Apply automata to formal input validation and explain what a regex actually computes


## 🔣 Symbol Primer

| Symbol | Name | Read it as | Meaning |
|---|---|---|---|
| $\Sigma$ | Alphabet | "sigma" | A finite non-empty set of input symbols |
| $\Sigma^*$ | Kleene star | "sigma star" | All finite strings over $\Sigma$, including the empty string $\varepsilon$ |
| $\varepsilon$ | Epsilon | "epsilon" | The empty string — length zero |
| $Q$ | State set | "Q" | Finite set of states |
| $\delta$ | Transition function | "delta" | $\delta: Q \times \Sigma \to Q$ — deterministic transitions |
| $q_0$ | Start state | "q-zero" | The initial state |
| $F$ | Accept states | "F" | $F \subseteq Q$ — states where the automaton accepts |
| $\hat{\delta}$ | Extended transition | "delta-hat" | $\hat{\delta}(q, w)$ — state reached from $q$ after reading string $w$ |
| $L(M)$ | Language of $M$ | "language of M" | Set of all strings accepted by automaton $M$ |
| $2^Q$ | Power set of $Q$ | "two to the Q" | Set of all subsets of $Q$ — states of the DFA from subset construction |

> **The connection to earlier modules.** A finite automaton is a directed graph
> (Module 04) where vertices are states and labelled edges are transitions.
> The language $L(M)$ is a set (Module 03). Accepting and rejecting a string
> is a Boolean decision. The formal 5-tuple definition connects automata directly
> to the relational and functional vocabulary of Module 03.

---


## 1 · Deterministic Finite Automata

A **DFA** $M = (Q, \Sigma, \delta, q_0, F)$ consists of:

- $Q$ — a finite set of **states**
- $\Sigma$ — the **alphabet** (finite set of input symbols)
- $\delta: Q \times \Sigma \to Q$ — the **transition function**: given current state and next symbol, returns next state
- $q_0 \in Q$ — the **start state**
- $F \subseteq Q$ — the set of **accepting states**

The DFA reads an input string one symbol at a time, transitioning between states
according to $\delta$. It **accepts** the string if it ends in an accepting state;
otherwise it **rejects**.

### Extended Transition Function

We extend $\delta$ to strings inductively:

$$\hat{\delta}(q, \varepsilon) = q$$
$$\hat{\delta}(q, wa) = \delta(\hat{\delta}(q, w), a) \quad \text{for } w \in \Sigma^*, a \in \Sigma$$

The language recognised by $M$ is:

$$L(M) = \{w \in \Sigma^* \mid \hat{\delta}(q_0, w) \in F\}$$

### Connection to Bounded Memory

A DFA with $|Q|$ states has exactly $|Q|$ possible "memory configurations" — it
cannot distinguish between inputs whose processing has led to the same state.
This bounded memory is both the DFA's strength (efficient implementation) and
its limitation (it cannot count arbitrarily, match balanced brackets, etc.).

**Security relevance.** Any system whose correctness depends on tracking a bounded
number of distinct situations — a protocol with a fixed number of phases, a rate
limiter with a fixed counter threshold, a login flow with bounded retry count —
can be modelled as a DFA. If the number of distinct situations is unbounded
(e.g., matching arbitrarily nested structures), a DFA is insufficient.

### Example — Input Validation DFA

A simple DFA over $\Sigma = \{0,1,2,\ldots,9,\texttt{.}\}$ that accepts
valid IPv4 address formats (simplified):

- $q_0$: start, expecting first digit group
- $q_1$: reading digit group, valid so far
- $q_2$: read a dot, expecting next digit group
- $q_3$: error state (invalid input seen)
- $F = \{q_1\}$: accept after a complete valid group

A full IPv4 validator would need more states to count the four groups, but the
principle is identical: the DFA's state encodes "where am I in the validation process."


## 2 · Non-Deterministic Finite Automata

An **NFA** $M = (Q, \Sigma, \delta, q_0, F)$ has the same components as a DFA
except the transition function allows:

$$\delta: Q \times (\Sigma \cup \{\varepsilon\}) \to 2^Q$$

Instead of one next state, $\delta$ returns a **set** of possible next states
(including the empty set — no valid transition). The NFA also permits
**$\varepsilon$-transitions** that change state without consuming any input symbol.

### Non-Determinism as Parallel Execution

Think of an NFA as exploring all possible computation paths simultaneously.
A string $w$ is **accepted** if *at least one* path through the NFA ends in an
accepting state:

$$L(M) = \{w \in \Sigma^* \mid \hat{\delta}(\{q_0\}, w) \cap F \neq \emptyset\}$$

### Why NFAs Matter for Security

**NFAs are the natural description of regular expressions.** Every regex
(`a(b|c)*d`, `\d{3}-\d{4}`, `[A-Z][a-z]+`) compiles directly to an NFA via
Thompson's construction. The NFA then processes input by exploring all possible
paths — and this exploration is where **ReDoS** vulnerabilities arise
(the subject of Unit 03).

**The DFA/NFA equivalence.** Despite the apparent extra power of non-determinism,
DFAs and NFAs recognise **exactly the same class of languages** — the regular
languages. The Subset Construction converts any NFA to an equivalent DFA,
at the cost of potentially exponentially more states.


## 3 · Subset Construction — Converting NFA to DFA

The **Subset Construction** (also called the Powerset Construction) converts an
NFA $N = (Q_N, \Sigma, \delta_N, q_0, F_N)$ to an equivalent DFA
$D = (Q_D, \Sigma, \delta_D, q_{D_0}, F_D)$:

- $Q_D = 2^{Q_N}$ — each DFA state is a **subset** of NFA states
- $q_{D_0} = \varepsilon\text{-closure}(\{q_0\})$ — start with all states reachable via $\varepsilon$ from $q_0$
- $\delta_D(S, a) = \varepsilon\text{-closure}\left(\bigcup_{q \in S} \delta_N(q, a)\right)$ — follow all NFA transitions, then close under $\varepsilon$
- $F_D = \{S \in Q_D \mid S \cap F_N \neq \emptyset\}$ — DFA accepts iff any NFA state in the subset is accepting

### The State Explosion

An NFA with $n$ states can produce a DFA with up to $2^n$ states. In the worst
case this blowup is unavoidable. However, in practice (and especially for most
regular expressions), the DFA has far fewer than $2^n$ reachable states.

**Security implication.** A regex engine that converts an NFA to a DFA before
matching is immune to ReDoS — the DFA processes each character in exactly one
step with no backtracking. However, the $2^n$ state explosion during compilation
can itself be a problem for complex patterns. Modern secure regex libraries
(Rust's `regex` crate, RE2) use DFA-based or NFA simulation approaches that
guarantee linear-time matching at the cost of potentially larger compilation.


## 4 · Regular Languages and Their Security Role

The class of languages recognised by finite automata is called the class of
**regular languages**. It has three equivalent characterisations:

| Characterisation | Description |
|---|---|
| **DFA/NFA** | Accepted by some finite automaton |
| **Regular expression** | Described by a regex using $\cup$, concatenation, $*$ |
| **Regular grammar** | Generated by a right-linear grammar |

All three describe the same class — this is the central theorem of automata theory.

### What Regular Languages Can and Cannot Do

**Can express:**
- Fixed-format strings: valid email addresses, IP addresses, phone numbers
- Patterns with bounded repetition: exactly 3 digits, 1–4 lowercase letters
- Input that belongs to a finite set

**Cannot express (require at least a pushdown automaton):**
- $\{a^n b^n \mid n \geq 0\}$ — matched pairs (balanced brackets, HTML tags)
- $\{ww \mid w \in \{a,b\}^*\}$ — strings that are their own repetition

**Security implication.** Regular expressions are appropriate for input validation
of fixed-format fields (API keys, email addresses, credit card numbers). They are
*not* appropriate for validating structured data with nesting (JSON, XML, code).
Using regex to validate nested structures is both incorrect (it may accept invalid
inputs) and potentially dangerous (ReDoS on complex patterns).


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[PV\]

| Automata concept | Security meaning |
|---|---|
| **State $q \in Q$** | The system's current "situation" — protocol phase, validation position |
| **Transition $\delta(q, a)$** | System response to receiving input/message $a$ in state $q$ |
| **Accepting state $F$** | Successful completion — valid input, completed handshake |
| **Error/trap state** | A non-accepting sink — invalid input detected, protocol violated |
| **DFA** | Deterministic, linear-time input validation — safe for production |
| **NFA** | Efficient description of complex patterns — backend for regex compilation |
| **Subset construction** | NFA → DFA conversion; linear matching at cost of more states |
| **Regular language** | The class of patterns safely expressible with regex |
| **Not regular** | Nested/recursive structure — use a parser, not a regex |
| **$2^n$ state explosion** | Compilation cost of converting complex NFAs to DFAs |

**The input validation principle:** use a DFA (or DFA-equivalent engine) for
all security-critical input validation. Never use a backtracking NFA engine on
untrusted input without careful analysis of catastrophic backtracking risk.

---


## Python: Visualization & Verification

**1 · DFA Simulator** — implement a DFA from its 5-tuple description, trace
any input string step by step, and visualise the state diagram as a directed graph.

**2 · NFA to DFA — Subset Construction** — implement the subset construction,
convert a small NFA to its DFA equivalent, and visualise both automata side by side.

**3 · Input Validation DFA** — build a DFA for validating a realistic input format
(IPv4 address skeleton); run a battery of valid and invalid inputs; produce an
acceptance report.


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)
print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from collections import deque

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')


### 1 · DFA Simulator and State Diagram

We implement a general DFA class, build a DFA that validates a simple
binary string format ("must start with 0 and end with 1"), trace several
inputs through it step by step, and draw the state diagram as a directed graph.


In [ ]:
# ── 1 · DFA Simulator ────────────────────────────────────────────────────────

class DFA:
    """
    Deterministic Finite Automaton.
    
    Parameters
    ----------
    states   : set of state names
    alphabet : set of symbols
    delta    : dict {(state, symbol): next_state}
    start    : start state
    accept   : set of accepting states
    """
    def __init__(self, states, alphabet, delta, start, accept):
        self.states   = states
        self.alphabet = alphabet
        self.delta    = delta
        self.start    = start
        self.accept   = accept

    def run(self, string, verbose=False):
        """Run the DFA on a string. Returns (accepted, trace)."""
        state = self.start
        trace = [state]
        for symbol in string:
            if symbol not in self.alphabet:
                return False, trace  # undefined — reject
            state = self.delta.get((state, symbol), 'TRAP')
            trace.append(state)
            if state == 'TRAP':
                break
        accepted = state in self.accept
        if verbose:
            arrow = ' → '.join(str(s) for s in trace)
            result = '✓ ACCEPT' if accepted else '✗ REJECT'
            print(f"  Input '{string}': {arrow}  [{result}]")
        return accepted, trace

# ── DFA: accepts binary strings that start with 0 and end with 1 ───────────────
# States: q0=start, q1=saw_0, q2=saw_0...ending_1, q3=error
dfa_bin = DFA(
    states   = {'q0', 'q1', 'q2', 'q3'},
    alphabet = {'0', '1'},
    delta    = {
        ('q0', '0'): 'q1',  ('q0', '1'): 'q3',
        ('q1', '0'): 'q1',  ('q1', '1'): 'q2',
        ('q2', '0'): 'q1',  ('q2', '1'): 'q2',
        ('q3', '0'): 'q3',  ('q3', '1'): 'q3',
    },
    start  = 'q0',
    accept = {'q2'},
)

test_inputs = ['01', '001', '0101', '00111', '10', '1', '0', '0110', '011010101']
print("DFA: accepts binary strings starting with 0 and ending with 1")
print()
for s in test_inputs:
    dfa_bin.run(s, verbose=True)

# ── Draw state diagram ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
G = nx.MultiDiGraph()
G.add_nodes_from(dfa_bin.states)

# Group edges by (src, dst) to handle parallel edges
edge_labels = {}
for (src, sym), dst in dfa_bin.delta.items():
    key = (src, dst)
    edge_labels[key] = edge_labels.get(key, [])
    edge_labels[key].append(sym)
for (src, dst), syms in edge_labels.items():
    G.add_edge(src, dst)

pos = {'q0': (0, 0), 'q1': (2.5, 0), 'q2': (5, 0), 'q3': (2.5, -2)}

nc = [TS_GREEN if n in dfa_bin.accept else
      TS_RED   if n == 'q3' else
      TS_AMBER if n == dfa_bin.start else TS_BLUE
      for n in G.nodes()]

nx.draw_networkx_nodes(G, pos, ax=ax, node_color=nc, node_size=1000, alpha=0.92)
nx.draw_networkx_labels(G, pos, ax=ax, font_color='white',
                        font_size=10, font_weight='bold')

# Draw edges with labels
for (src, dst), syms in edge_labels.items():
    label = ','.join(sorted(syms))
    rad   = 0.25 if src == dst else (0.3 if src > dst else 0.0)
    nx.draw_networkx_edges(G, pos, ax=ax, edgelist=[(src, dst)],
                           arrows=True, arrowsize=20, width=2,
                           edge_color=TS_GRAY, alpha=0.8,
                           connectionstyle=f'arc3,rad={rad}')
    mx = (pos[src][0] + pos[dst][0]) / 2
    my = (pos[src][1] + pos[dst][1]) / 2 + (0.3 if rad > 0 else 0.2)
    ax.text(mx, my, label, ha='center', fontsize=10,
            color=TS_AMBER, fontweight='bold')

# Start arrow
ax.annotate('', xy=(pos['q0'][0]-0.1, pos['q0'][1]),
            xytext=(pos['q0'][0]-0.9, pos['q0'][1]),
            arrowprops=dict(arrowstyle='->', color=TS_GRAY, lw=2))

legend = [mpatches.Patch(color=TS_AMBER, label='Start state'),
          mpatches.Patch(color=TS_GREEN, label='Accept state'),
          mpatches.Patch(color=TS_RED,   label='Error/trap state'),
          mpatches.Patch(color=TS_BLUE,  label='Regular state')]
ax.legend(handles=legend, loc='upper right', fontsize=9)
ax.set_title('DFA: accepts binary strings starting with 0 ending with 1
'
             'Each edge label is the symbol that triggers that transition',
             pad=10, fontsize=11)
ax.axis('off'); ax.set_facecolor('white'); fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


**What do you see?**

The trace output shows each state transition, making the DFA's execution completely
transparent. The state diagram makes the structure clear: `q0` is the start,
`q1` is the "seen 0, accumulating" state, `q2` is the accepting state ("ends in 1
after starting with 0"), and `q3` is the error sink — once entered, never escaped.

Notice that inputs starting with 1 immediately enter `q3` and stay there — the
DFA has "remembered" that the first character was wrong, even though it processes
subsequent characters. This is bounded memory in action: the state encodes exactly
the information needed to make future decisions, nothing more.


### 2 · Subset Construction — NFA to DFA

We build a small NFA that accepts strings ending in `ab`, convert it to an
equivalent DFA using subset construction, and visualise both automata.


In [ ]:
# ── 2 · Subset Construction — NFA to DFA ─────────────────────────────────────

class NFA:
    """Non-deterministic Finite Automaton (without ε-transitions for clarity)."""
    def __init__(self, states, alphabet, delta, start, accept):
        self.states   = states
        self.alphabet = alphabet
        self.delta    = delta   # dict {(state, symbol): frozenset of next states}
        self.start    = start
        self.accept   = accept

    def subset_construction(self):
        """Convert NFA to equivalent DFA via subset construction."""
        start_set = frozenset([self.start])
        dfa_states = {start_set}
        dfa_delta  = {}
        queue      = deque([start_set])
        visited    = {start_set}

        while queue:
            S = queue.popleft()
            for sym in self.alphabet:
                # Compute the set of NFA states reachable from any state in S on sym
                next_S = frozenset(
                    ns for q in S
                    for ns in self.delta.get((q, sym), frozenset())
                )
                dfa_delta[(S, sym)] = next_S
                if next_S not in visited:
                    visited.add(next_S)
                    dfa_states.add(next_S)
                    queue.append(next_S)

        dfa_accept = frozenset(
            S for S in dfa_states if S & self.accept
        )
        return dfa_states, dfa_delta, start_set, dfa_accept

# NFA that accepts strings ending in "ab" over alphabet {a, b}
# States: 0=start, 1=saw_a, 2=saw_ab(accept)
nfa_ab = NFA(
    states   = {0, 1, 2},
    alphabet = {'a', 'b'},
    delta    = {
        (0, 'a'): frozenset({0, 1}),   # stay or start "ab" pattern
        (0, 'b'): frozenset({0}),
        (1, 'a'): frozenset({1}),
        (1, 'b'): frozenset({2}),       # complete "ab"
        (2, 'a'): frozenset({1}),
        (2, 'b'): frozenset({0}),
    },
    start  = 0,
    accept = frozenset({2}),
)

dfa_states, dfa_delta, dfa_start, dfa_accept = nfa_ab.subset_construction()

# State name mapping (frozenset → readable label)
state_names = {s: '{' + ','.join(str(x) for x in sorted(s)) + '}'
               if s else '∅' for s in dfa_states}

print("Subset Construction: NFA → DFA")
print(f"
NFA: {len(nfa_ab.states)} states  |  DFA: {len(dfa_states)} states")
print(f"
DFA states (each is a subset of NFA states):")
for S in sorted(dfa_states, key=lambda x: sorted(x)):
    name = state_names[S]
    kind = (' ← START' if S == dfa_start else '') +            (' ← ACCEPT' if S in dfa_accept else '')
    print(f"  {name}{kind}")

print(f"
DFA transition table:")
print(f"{'State':<20}  {'on a':>8}  {'on b':>8}")
print("─" * 40)
for S in sorted(dfa_states, key=lambda x: sorted(x)):
    sa = state_names.get(dfa_delta.get((S,'a'), frozenset()), '∅')
    sb = state_names.get(dfa_delta.get((S,'b'), frozenset()), '∅')
    print(f"  {state_names[S]:<18}  {sa:>8}  {sb:>8}")

# ── Verify equivalence ────────────────────────────────────────────────────────
def nfa_accepts(nfa, string):
    states = frozenset([nfa.start])
    for sym in string:
        states = frozenset(ns for q in states
                           for ns in nfa.delta.get((q, sym), frozenset()))
    return bool(states & nfa.accept)

def dfa_accepts_subset(dfa_d, dfa_st, dfa_ac, string):
    state = dfa_st
    for sym in string:
        state = dfa_d.get((state, sym), frozenset())
    return state in dfa_ac

test_strings = ['ab', 'aab', 'bab', 'ababab', 'ba', 'b', 'a', 'abab', 'aabb']
print(f"
Equivalence check (NFA == DFA for all test strings):")
all_match = True
for s in test_strings:
    nfa_res = nfa_accepts(nfa_ab, s)
    dfa_res = dfa_accepts_subset(dfa_delta, dfa_start, dfa_accept, s)
    match = nfa_res == dfa_res
    all_match = all_match and match
    print(f"  '{s}': NFA={'✓' if nfa_res else '✗'}  DFA={'✓' if dfa_res else '✗'}  {'✓' if match else '✗ MISMATCH'}")
print(f"
All equivalent: {all_match} ✓")


**What do you see?**

The subset construction produces DFA states that are subsets of NFA states —
$\{0\}$, $\{0,1\}$, $\{0,2\}$, $\{0,1,2\}$ etc. Each DFA state tracks all the
NFA states the machine might currently be in simultaneously, collapsing the
non-determinism into a deterministic process.

The equivalence check confirms that for every test string, the NFA and DFA
agree on acceptance — they recognise exactly the same language (strings ending
in "ab"). This is the Subset Construction theorem verified computationally.


### 3 · Input Validation DFA — IPv4 Address Skeleton

We build a DFA for a simplified IPv4 address validation and run a structured
battery of valid and invalid inputs through it, producing an acceptance report.


In [ ]:
# ── 3 · Input Validation DFA ─────────────────────────────────────────────────
#
# Simplified IPv4 skeleton validator:
# Accepts: strings of the form D.D.D.D where D is one or more digits
# Rejects: anything else

# Alphabet: digits 0-9 and '.'
DIGITS = set('0123456789')
ALPHABET = DIGITS | {'.'}

# States
# q0: start (expecting first digit)
# q1: reading a digit group (at least one digit seen)
# q2: saw a dot (expecting digit)
# qE: error/trap
# qA: accept (same as q1 but after exactly 3 dots)

# We'll use string state names for clarity
delta_ipv4 = {}
states_ipv4 = {'q0','q1','q2','qA','qE'}

def make_ipv4_dfa():
    delta = {}
    # q0: start — expecting first digit
    for d in DIGITS: delta[('q0', d)] = 'q1'
    delta[('q0', '.')] = 'qE'

    # q1: reading first group (0 dots seen)
    for d in DIGITS: delta[('q1', d)] = 'q1'
    delta[('q1', '.')] = 'q2_1'  # first dot

    # q2_1: after first dot — expecting digit
    for d in DIGITS: delta[('q2_1', d)] = 'q3_1'
    delta[('q2_1', '.')] = 'qE'

    # q3_1: reading second group
    for d in DIGITS: delta[('q3_1', d)] = 'q3_1'
    delta[('q3_1', '.')] = 'q2_2'

    # q2_2: after second dot
    for d in DIGITS: delta[('q2_2', d)] = 'q3_2'
    delta[('q2_2', '.')] = 'qE'

    # q3_2: reading third group
    for d in DIGITS: delta[('q3_2', d)] = 'q3_2'
    delta[('q3_2', '.')] = 'q2_3'

    # q2_3: after third dot
    for d in DIGITS: delta[('q2_3', d)] = 'qA'
    delta[('q2_3', '.')] = 'qE'

    # qA: reading fourth group (accept state)
    for d in DIGITS: delta[('qA', d)] = 'qA'
    delta[('qA', '.')] = 'qE'

    # qE: trap — absorb everything
    for sym in ALPHABET: delta[('qE', sym)] = 'qE'

    states = {'q0','q1','q2_1','q3_1','q2_2','q3_2','q2_3','qA','qE'}
    return DFA(
        states   = states,
        alphabet = ALPHABET,
        delta    = delta,
        start    = 'q0',
        accept   = {'qA'},
    )

ipv4_dfa = make_ipv4_dfa()

test_cases = [
    ('192.168.1.1',   True,  'valid IPv4'),
    ('10.0.0.255',    True,  'valid IPv4'),
    ('1.2.3.4',       True,  'valid IPv4'),
    ('0.0.0.0',       True,  'valid (all zeros)'),
    ('999.999.999.999',True,  'syntactically valid (values not checked)'),
    ('192.168.1',     False, 'missing last group'),
    ('192.168.1.1.5', False, 'too many groups'),
    ('192.168..1',    False, 'double dot'),
    ('.192.168.1.1',  False, 'leading dot'),
    ('192.168.1.',    False, 'trailing dot'),
    ('192.168.a.1',   False, 'letter in group'),
    ('',              False, 'empty string'),
]

print("IPv4 Address Skeleton Validator (simplified — no range check)")
print(f"{'Input':<22} {'Expected':>10} {'Result':>8} {'Match':>7}")
print("─" * 55)
correct = 0
for inp, expected, note in test_cases:
    accepted, _ = ipv4_dfa.run(inp)
    match = accepted == expected
    correct += match
    symbol = '✓' if match else '✗'
    print(f"  {inp:<20}  {str(expected):>10}  {'ACCEPT' if accepted else 'REJECT':>8}  "
          f"{symbol:>6}  {note}")

print(f"
Accuracy: {correct}/{len(test_cases)} correct")
print()
print("Note: This DFA validates FORMAT only — not value ranges.")
print("  '999.999.999.999' is syntactically valid (correct format)")
print("  but semantically invalid (each group must be 0-255).")
print("  A full validator would need additional range checking logic.")


**What do you see?**

The validator correctly accepts all well-formed IPv4-format strings and rejects
all malformed ones. The important note at the bottom illustrates a fundamental
limitation of finite automata: they can check **syntactic** constraints (format)
but not all **semantic** ones (value ranges).

Checking that each octet is between 0 and 255 is a bounded counting task — it
requires comparing a multi-digit number to 255. While technically expressible
as a DFA (with many states for each digit position), it would be unwieldy.
A practical validator combines a DFA for format with arithmetic checks for
range — the DFA handles the regular structure, arithmetic handles what automata
do awkwardly.

This is the lesson: understand what your validation tool can and cannot express.
Using regex for structure, supplemented by explicit range checks, is the correct
architecture for IPv4 validation.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. MINIMISATION
#    A DFA may have redundant states — states that are equivalent (accept the
#    same future strings). Hopcroft's algorithm minimises a DFA.
#    For the binary string DFA from Section 1, check whether all states are
#    distinguishable. (Two states q, r are distinguishable if there exists a
#    string that leads to acceptance from q but not r, or vice versa.)
#    Is the DFA already minimal? If not, merge equivalent states.
#
# 2. COMPLEMENT AND INTERSECTION
#    The complement of a DFA (swap accept ↔ non-accept states) recognises
#    the complement language. Build the complement of the IPv4 validator —
#    a DFA that accepts INVALID IPv4 strings.
#    Verify: every string accepted by the complement is rejected by the original.
#    Then build the intersection (strings accepted by BOTH) — it should be empty.
#
# 3. YOUR OWN VALIDATOR
#    Build a DFA for one of these formats:
#    (a) API keys: exactly 32 hex characters [0-9a-fA-F]
#    (b) Semantic version: MAJOR.MINOR.PATCH where each is one or more digits
#    (c) JWT format: base64url.base64url.base64url (three groups separated by dots,
#        each group containing only [A-Za-z0-9_-] and = padding)
#    Draw the state diagram and run at least 10 test cases.

# Your work here:


---

## Summary

| Concept | Definition | Security meaning |
|---|---|---|
| **DFA $(Q,\Sigma,\delta,q_0,F)$** | 5-tuple with deterministic transitions | Any system with bounded state and deterministic response |
| **NFA** | DFA with non-deterministic / $\varepsilon$ transitions | Natural description of regex patterns |
| **$L(M)$** | Set of accepted strings | The language the automaton validates |
| **Subset construction** | Converts NFA → equivalent DFA | Makes regex matching deterministic and linear |
| **Regular language** | Recognisable by DFA/NFA/regex | Safe for fixed-format input validation |
| **Not regular** | Nested/recursive structure | Use a parser — regex is the wrong tool |
| **Trap/error state** | Non-accepting sink | Detected invalid input; no recovery |
| **$2^n$ state blowup** | Worst-case DFA size from $n$-state NFA | Compilation cost of complex regex patterns |

---

## Up Next

**Module 09 · Unit 02 — Protocol State Machines**

We lift the automaton model to protocol verification. A protocol is a finite
automaton where states are protocol phases and transitions are message exchanges.
The MCP handshake is modelled formally, its invariants stated, and its violation
conditions derived. The Module 00 invariant promise is finally delivered in full:
a protocol state machine with an unreachable error state is formally proved
safe using the automaton structure.

$\rightarrow$ `modules/module-09/unit-02-protocol-state-machines.ipynb`
